# Track 2: Is the 500MB Compressible?

Track 1 (Sprint/Marathon) ships as-is. This is speculative R&D.

Measures whether the inter-layer V matrices have exploitable structure
for T9-style delta-encode-then-unfold compression.

Four measurements:
1. Adjacent layer cosine similarity (are layers similar?)
2. Delta norms (how much does V change per layer?)
3. PCA dimensionality (what subspace do all V matrices live in?)
4. Transformation consistency (is there one T that maps every layer to the next?)

In [ ]:
# Cell 1: Setup + ensure teacher spectra exist
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/realityinspector/nanogpt-prism.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
!python data/shakespeare_char/prepare.py 2>/dev/null

# Train teacher if needed
if not os.path.exists('.prism_cache/shakespeare/directions.pt'):
    print('Training teacher...')
    subprocess.run(['python', 'train.py', 'config/train_shakespeare_char.py',
        '--max_iters=2000', '--eval_interval=2000', '--eval_iters=50',
        '--log_interval=2000', '--out_dir=out-teacher',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False'], capture_output=True, timeout=600)
    subprocess.run(['python', 'prism_extract.py', '--ckpt', 'out-teacher/ckpt.pt',
        '--out', '.prism_cache/shakespeare'], capture_output=True, timeout=120)
print('Ready.')

In [ ]:
# Cell 2: Run the measurement
!python measure_v_structure.py --cache .prism_cache/shakespeare

In [ ]:
# Cell 3: Download results
from google.colab import files
files.download('.prism_cache/shakespeare/v_structure.json')